In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# محلّل التحسين: دالة Qiskit من Q-CTRL Fire Opal
> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي IBM Quantum&reg; Premium Plan وFlex Plan وOn-Prem (عبر IBM Quantum Platform API) Plan. هي في مرحلة إصدار معاينة وقابلة للتغيير.

## نظرة عامة
مع محلّل التحسين Fire Opal، يمكنك حلّ مسائل التحسين على النطاق الاستخدامي على الأجهزة الكمومية دون الحاجة إلى خبرة كمومية. ما عليك سوى إدخال تعريف المسألة رفيع المستوى، ويتولى المحلّل الباقي. سير العمل بالكامل يدرك الضجيج ويستفيد من [إدارة أداء Fire Opal](/guides/q-ctrl-performance-management) في الخلفية. يقدّم المحلّل باستمرار حلولاً دقيقة للمسائل الصعبة كلاسيكيًا، حتى على نطاق الجهاز الكامل لأكبر أجهزة IBM&reg; QPU.

المحلّل مرن ويمكن استخدامه لحلّ مسائل التحسين التوافقية المعرَّفة كدوال هدف أو رسوم بيانية عشوائية. لا يجب ربط المسائل بطبولوجيا الجهاز. المسائل غير المقيدة والمقيدة قابلة للحلّ، بشرط أن تُصاغ القيود كحدود عقوبة. الأمثلة المضمّنة في هذا الدليل توضّح كيفية حلّ مسألة تحسين غير مقيدة ومسألة مقيدة على النطاق الاستخدامي باستخدام أنواع مدخلات مختلفة للمحلّل. المثال الأول يتضمن مسألة Max-Cut معرَّفة على رسم بياني منتظم ثلاثي 156 عقدة، بينما يعالج المثال الثاني مسألة Minimum Vertex Cover من 50 عقدة معرَّفة بدالة تكلفة.

للحصول على إمكانية الوصول إلى محلّل التحسين، [تواصل مع Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).

## وصف الدالة
يحسّن المحلّل الخوارزمية بالكامل ويؤتمتها، من قمع الأخطاء على مستوى الأجهزة إلى تعيين المسائل بكفاءة والتحسين الكلاسيكي في حلقة مغلقة. خلف الكواليس، تقلّل خطوط أنابيب المحلّل الأخطاء في كل مرحلة، مما يُمكّن الأداء المحسّن المطلوب للتوسع الفعلي. سير العمل الأساسي مستوحى من خوارزمية التحسين الكمومي التقريبي (QAOA)، وهي خوارزمية هجينة كمومية-كلاسيكية. للاطلاع على ملخص تفصيلي لسير عمل محلّل التحسين الكامل، راجع [المخطوطة المنشورة](https://arxiv.org/abs/2406.01743).

![تصوّر لسير عمل محلّل التحسين](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

لحلّ مسألة عامة مع محلّل التحسين:
1. عرِّف مسألتك كدالة هدف، أو رسم بياني، أو سلسلة دوران `SparsePauliOp`.
2. اتصل بالدالة عبر كتالوج دوال Qiskit.
3. شغّل المسألة مع المحلّل واسترجع النتائج.

## المدخلات والمخرجات
### المدخلات
| الاسم    | النوع                    | الوصف                                                                                                                                                                                         | مطلوب | الافتراضي | مثال |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` أو `SparsePauliOp` | أحد التمثيلات المدرجة ضمن "صيغ المسائل المقبولة"                                                                                                                                                  | نعم | غير متاح   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | اسم فئة المسألة؛ يُستخدم فقط لتعريفات مسائل الرسم البياني وسلسلة الدوران، المحدودة بـ"maxcut" أو "spin_chain"؛ غير مطلوب لتعريفات مسائل دالة الهدف العشوائية | لا      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | اسم الخلفية المراد استخدامها                                                                                                                                                                          | لا  | أقل الخلفيات ازدحامًا التي يمكن لمثيلك الوصول إليها    | `"ibm_fez"`      |
| options  | `dict`                  | خيارات الإدخال، بما في ذلك: (اختياري) `session_id`: `str`؛ السلوك الافتراضي ينشئ جلسة جديدة                                                                                      | لا | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**صيغ المسائل المقبولة:**
- تمثيل التعبير متعدد الحدود لدالة هدف. يُنصح بإنشائه في Python بكائن SymPy Poly موجود وتنسيقه كسلسلة نصية باستخدام [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- تمثيل رسم بياني لنوع مسألة محدد. يجب إنشاء الرسم البياني باستخدام مكتبة networkx في Python، ثم تحويله إلى سلسلة نصية باستخدام دالة networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- تمثيل سلسلة الدوران لمسألة محددة. يجب تمثيل سلسلة الدوران ككائن `SparsePauliOp`؛ راجع [التوثيق](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) لمزيد من التفاصيل.

**الخلفيات المدعومة:**
شغّل الكود التالي لرؤية قائمة الخلفيات المدعومة حاليًا. إذا لم تكن جهازك مدرجًا، [تواصل مع Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) لإضافة الدعم.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**الخيارات:**
| الاسم   | النوع   | الوصف  | الافتراضي |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | معرّف جلسة Qiskit Runtime موجودة  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | قائمة بوسوم الوظائف | `[]`|

### المخرجات
| الاسم    | النوع | الوصف | مثال |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | الحلّ والبيانات الوصفية المدرجة ضمن "محتويات قاموس النتيجة"         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**محتويات قاموس النتيجة:**
| الاسم    | النوع | الوصف |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | أفضل تكلفة دنيا عبر جميع التكرارات        |
| final_bitstring_distribution  | `CountsDict`              | قاموس أعداد السلسلة الثنائية المرتبط بالتكلفة الدنيا        |
| solution_bitstring | `str`              | السلسلة الثنائية ذات أفضل تكلفة في التوزيع النهائي         |
| iteration_count  | `int`              | العدد الكلي لتكرارات QAOA التي أجراها المحلّل        |
| variables_to_bitstring_index_map  | `float`              | الربط بين المتغيرات والبت المقابل في السلسلة الثنائية       |
| best_parameters  | `list[float]`            | معاملات beta وgamma المحسّنة عبر جميع التكرارات  |
| warnings  |`list[str]`              | التحذيرات الصادرة أثناء تجميع QAOA أو تشغيله (تعود افتراضيًا بـNone)   |

## المعايير المرجعية
تُظهر [نتائج المعايير المرجعية المنشورة](https://arxiv.org/abs/2406.01743) أن المحلّل ينجح في حلّ مسائل تتجاوز 120 كيوبت، متفوقًا حتى على النتائج المنشورة سابقًا على أجهزة التليين الكمومي والأيونات المحاصرة. مقاييس المعايير المرجعية التالية تعطي مؤشرًا تقريبيًا على دقة أنواع المسائل وقابليتها للتوسع بناءً على بعض الأمثلة. قد تختلف المقاييس الفعلية بناءً على خصائص المسألة المختلفة، مثل عدد الحدود في دالة الهدف (الكثافة) وموقعها، وعدد المتغيرات، والرتبة متعددة الحدود.

"عدد الكيوبتات" المشار إليه ليس حدًا صارمًا بل يمثّل عتبات تقريبية حيث يمكن توقع دقة حلول عالية الاتساق. مسائل أكبر حجمًا تمّ حلّها بنجاح، ويُشجّع الاختبار خارج هذه الحدود.

الاتصالية العشوائية بين الكيوبتات مدعومة عبر جميع أنواع المسائل.

| نوع المسألة    | عدد الكيوبتات | مثال | الدقة | الوقت الكلي (ث) | استخدام وقت التشغيل (ث) | عدد التكرارات
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| مسائل تربيعية متفرقة الاتصال  | 156 | Max-Cut ثلاثي منتظم| 100%     | 1764     | 293          | 16 |
| تحسين ثنائي ذو رتبة عليا | 156 | نموذج Ising spin-glass | 100%      | 1461     | 272          | 16 |
| مسائل تربيعية كثيفة الاتصال | 50 | Max-Cut متكامل الاتصال| 100%      |  1758    | 268  | 12 |
| مسألة مقيدة بحدود عقوبة | 50 | Minimum Vertex Cover موزون بكثافة حواف 8% | 100%      | 1074     | 215 | 10 |

## البدء
أولًا، سجّل الدخول باستخدام [مفتاح IBM Quantum API](http://quantum.cloud.ibm.com/). ثم اختر دالة Qiskit كما يلي. (هذا المقتطف يفترض أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) في بيئتك المحلية مسبقًا.)

In [2]:
# %pip install networkx numpy

## مثال: تحسين غير مقيد
شغّل مسألة [القطع الأقصى](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). يوضّح المثال التالي قدرات المحلّل على مسألة Max-Cut لرسم بياني غير موزون منتظم ثلاثي من 156 عقدة، لكن يمكنك أيضًا حلّ مسائل الرسم البياني الموزون.
بالإضافة إلى `qiskit-ibm-catalog`، ستحتاج أيضًا إلى الحزم التالية لتشغيل هذا المثال: `networkx` و`numpy`. يمكنك تثبيت هذه الحزم بإلغاء تعليق الخلية التالية إذا كنت تشغّل هذا المثال في notebook باستخدام نواة IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Output of the previous code cell](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

يقبل المحلّل سلسلة نصية كمدخل لتعريف المسألة.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


تحقق من [حالة](/guides/functions#check-job-status) حمل عمل دالة Qiskit أو استرجع [النتائج](/guides/functions#retrieve-results) كما يلي:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. استرجاع النتيجة
استرجع قيمة القطع المثلى من قاموس النتائج.

> **Note:** ربما تغيّر تعيين المتغيرات إلى السلسلة الثنائية. يحتوي قاموس الإخراج على قاموس فرعي `variables_to_bitstring_index_map` يساعد في التحقق من الترتيب.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

يمكنك التحقق من دقة النتيجة بحلّ المسألة كلاسيكيًا باستخدام محلّلات مفتوحة المصدر مثل [PuLP](https://coin-or.github.io/pulp/) إذا لم يكن الرسم البياني كثيف الاتصال. قد تتطلب مسائل الكثافة العالية محلّلات كلاسيكية متقدمة للتحقق من صحة الحلّ.

## مثال: تحسين مقيد
مثال Max-Cut السابق هو مسألة تحسين ثنائي تربيعي غير مقيد شائعة. يمكن استخدام محلّل التحسين من Q-CTRL لأنواع مسائل متعددة، بما في ذلك التحسين المقيد. يمكنك حلّ أنواع مسائل عشوائية بإدخال تعريف المسألة ممثَّلًا كمتعدد حدود حيث تُنمذَج القيود كحدود عقوبة.

يوضّح المثال التالي كيفية بناء دالة تكلفة لمسألة تحسين مقيدة، [الغطاء الأدنى للرؤوس](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
بالإضافة إلى حزمتي `qiskit-ibm-catalog` و`qiskit`، ستحتاج أيضًا إلى الحزم التالية لتشغيل هذا المثال: `numpy` و`networkx` و`sympy`. يمكنك تثبيت هذه الحزم بإلغاء تعليق الخلية التالية إذا كنت تشغّل هذا المثال في notebook باستخدام نواة IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. تعريف المسألة
عرِّف مسألة MVC عشوائية بتوليد رسم بياني بعقد موزونة عشوائيًا.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Output of the previous code cell](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

يمكن صياغة نموذج التحسين القياسي للـMVC الموزون كما يلي. أولًا، يجب إضافة عقوبة لأي حالة لا تكون فيها حافة متصلة برأس في المجموعة الفرعية. لذا، ليكن $n_i = 1$ إذا كان الرأس $i$ في الغطاء (أي في المجموعة الفرعية) و$n_i = 0$ في الحالة المعاكسة. ثانيًا، الهدف هو تقليل العدد الكلي للرؤوس في المجموعة الفرعية، ويمكن تمثيله بالدالة التالية:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

الآن يجب أن تشمل كل حافة في الرسم البياني على الأقل نقطة نهاية واحدة من الغطاء، ويمكن التعبير عن ذلك بالمتراجحة:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

يجب معاقبة أي حالة لا تكون فيها الحافة متصلة برأس من الغطاء. يمكن تمثيل ذلك في دالة التكلفة بإضافة عقوبة من الشكل $P(1-n_i-n_j+n_i n_j)$ حيث $P$ ثابت عقوبة موجب. وبذلك، البديل غير المقيد للمتراجحة المقيدة للـMVC الموزون هو:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. تشغيل المسألة